# 7장 실습 ② — 시험 데이터를 볼 때마다 생기는 일

**PyTorch 판**

*"test를 validation으로 쓰면 안 된다"* — 왜 안 되는지를 숫자로 봅니다.

> 본문 §7.6에서, 초고의 가설(*"성능이 크게 부풀려진다"*)이 실측과 달랐습니다.
> 이 노트북이 그 실측입니다. **왜 안 나왔는지, 그리고 진짜 원인은 무엇인지**를
> 두 실험으로 갈라냅니다.

## 7.0 준비

In [1]:
try:
    import dlbook
except ImportError:
    !pip install -q "dlbook @ git+https://github.com/dhrim/deep-learning-in-one-semester.git"
    import dlbook

In [2]:
import numpy as np
import matplotlib.pyplot as plt

import dlbook
from dlbook import data, metrics, plot

dlbook.set_seed(42)
plot.use_korean()
print(dlbook.versions())

{'python': '3.12.3', 'numpy': '2.1.3', 'keras': '-', 'tensorflow': '-', 'torch': '2.14.0', 'keras_backend': '-'}


## 7.1 순수한 선택 편향 — 실력이 똑같아도 생깁니다

먼저 모델 없이 확인합니다. **실제 성능이 전부 같은 후보들** 중에서
시험 데이터로 가장 좋은 것을 고르면 어떻게 되는지 봅니다.

In [3]:
# 실제 성능이 **전부 0.750으로 똑같은** 후보들을 놓고,
# 시험 데이터에서 가장 좋아 보이는 것을 고르면 얼마나 부풀려지는가.
# 운으로만 갈리는 상황이므로, 나온 값이 곧 **선택 편향의 크기**다.
rng = np.random.default_rng(0)
p_true = 0.75
ks = [1, 5, 20] if dlbook.smoke.is_smoke() else [1, 5, 20, 100]
ns = [100, 250, 1000] if dlbook.smoke.is_smoke() else [100, 250, 1000, 5000]
trials = 1000 if dlbook.smoke.is_smoke() else 4000

print(f"{'시험 데이터':>10}" + "".join(f"{'후보 ' + str(k) + '개':>12}" for k in ks))
for n_test in ns:
    row = []
    for k in ks:
        obs = rng.binomial(n_test, p_true, size=(trials, k)) / n_test
        bias = obs.max(axis=1).mean() - p_true
        row.append(bias)
        dlbook.record(f"ch07_bias_n{n_test}_k{k}", bias)
    print(f"{n_test:>10}" + "".join(f"{v:>+12.3f}" for v in row))

print()
print("→ 후보가 하나면 부풀림이 없습니다. 한 번만 재고 보고하면 정직한 숫자입니다.")
print("→ 후보가 많을수록, 시험 데이터가 작을수록 커집니다.")
print("→ 여러분은 한 학기에 시험 데이터를 몇 번 봅니까?")

    시험 데이터       후보 1개       후보 5개      후보 20개     후보 100개
ch07_bias_n100_k1 = 0.0006
ch07_bias_n100_k5 = 0.0492
ch07_bias_n100_k20 = 0.0781
ch07_bias_n100_k100 = 0.1035
       100      +0.001      +0.049      +0.078      +0.104
ch07_bias_n250_k1 = -0.0002
ch07_bias_n250_k5 = 0.0318
ch07_bias_n250_k20 = 0.0498
ch07_bias_n250_k100 = 0.0666
       250      -0.000      +0.032      +0.050      +0.067
ch07_bias_n1000_k1 = 0.0002
ch07_bias_n1000_k5 = 0.0161
ch07_bias_n1000_k20 = 0.0252
ch07_bias_n1000_k100 = 0.0339
      1000      +0.000      +0.016      +0.025      +0.034
ch07_bias_n5000_k1 = -0.0001
ch07_bias_n5000_k5 = 0.0071
ch07_bias_n5000_k20 = 0.0114
ch07_bias_n5000_k100 = 0.0152
      5000      -0.000      +0.007      +0.011      +0.015

→ 후보가 하나면 부풀림이 없습니다. 한 번만 재고 보고하면 정직한 숫자입니다.
→ 후보가 많을수록, 시험 데이터가 작을수록 커집니다.
→ 여러분은 한 학기에 시험 데이터를 몇 번 봅니까?


## 7.2 학습 함수 — 여기만 판마다 다릅니다

In [4]:
import torch, itertools
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

def build_configs():
    units = [(32, 32), (64, 64), (128, 128)] if dlbook.smoke.is_smoke() else \
            [(32, 32), (64, 64), (128, 128), (256, 256), (64, 64, 64)]
    return list(itertools.product(units, [0.0003, 0.001, 0.003], [0.0, 0.3]))

def fit_and_score(cfg, xr, yr, train_i, eval_is):
    """설정 하나를 학습시키고, 여러 조각에서의 정확도를 돌려준다."""
    units, lr, dr = cfg
    dlbook.set_seed(42)
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    ls, prev = [], 2
    for u in units:
        ls += [nn.Linear(prev, u), nn.ReLU()]
        if dr:
            ls.append(nn.Dropout(dr))
        prev = u
    ls.append(nn.Linear(prev, 1))
    m = nn.Sequential(*ls).to(dev)
    opt = torch.optim.Adam(m.parameters(), lr=lr)
    crit = nn.BCEWithLogitsLoss()
    dl = DataLoader(TensorDataset(torch.tensor(xr[train_i], dtype=torch.float32),
                                  torch.tensor(yr[train_i], dtype=torch.float32).view(-1, 1)),
                    batch_size=32, shuffle=True)
    for _ in range(dlbook.smoke.epochs(120)):
        m.train()
        for xb, yb in dl:
            xb, yb = xb.to(dev), yb.to(dev)
            opt.zero_grad(); crit(m(xb), yb).backward(); opt.step()
    m.eval()
    out = []
    for i in eval_is:
        with torch.no_grad():
            lg = m(torch.tensor(xr[i], dtype=torch.float32).to(dev))
        out.append(metrics.accuracy(yr[i], (lg.cpu().numpy().reshape(-1) > 0).astype("int64")))
    return out

## 7.3 진짜 모델로 확인합니다

데이터를 **넷**으로 나눕니다. `holdout` 은 끝까지 아무도 보지 않습니다 —
'진짜 성능'을 재기 위한 것입니다.

In [5]:
# 이번엔 진짜 모델로. 데이터를 **넷**으로 나눈다.
# holdout은 끝까지 아무도 보지 않는다 — '진짜 성능'을 재기 위한 것이다.
xr, yr = data.spirals(1000, seed=42, noise=0.28)
rng = np.random.default_rng(0)
idx = rng.permutation(len(xr))
q = len(xr) // 4
holdout_i, test_i, val_i, train_i = idx[:q], idx[q:2*q], idx[2*q:3*q], idx[3*q:]

configs = build_configs()          # 판마다 다른 셀에서 정의한다
print(f"후보 설정 {len(configs)}개를 전부 학습시킨 뒤, 하나를 고른다.")

rows = []
for cfg in configs:
    a = fit_and_score(cfg, xr, yr, train_i, [val_i, test_i, holdout_i])
    rows.append(a)
rows = np.array(rows)

bv, bt = int(rows[:, 0].argmax()), int(rows[:, 1].argmax())
print()
print(f"{'고르는 기준':<22}{'보고할 성능':>12}{'실제(holdout)':>16}{'부풀림':>10}")
print(f"{'검증(val)으로 고름':<22}{rows[bv,1]:>12.3f}{rows[bv,2]:>16.3f}{rows[bv,1]-rows[bv,2]:>+10.3f}")
print(f"{'시험(test)으로 고름':<22}{rows[bt,1]:>12.3f}{rows[bt,2]:>16.3f}{rows[bt,1]-rows[bt,2]:>+10.3f}")
dlbook.record("ch07_pick_by_val_inflation", rows[bv,1] - rows[bv,2])
dlbook.record("ch07_pick_by_test_inflation", rows[bt,1] - rows[bt,2])
print()
print(f"후보들의 시험 성능: 최소 {rows[:,1].min():.3f}  최대 {rows[:,1].max():.3f}"
      f"  표준편차 {rows[:,1].std():.3f}")

후보 설정 30개를 전부 학습시킨 뒤, 하나를 고른다.



고르는 기준                      보고할 성능     실제(holdout)       부풀림
검증(val)으로 고름                 0.760           0.768    -0.008
시험(test)으로 고름                0.788           0.744    +0.044
ch07_pick_by_val_inflation = -0.0080
ch07_pick_by_test_inflation = 0.0440

후보들의 시험 성능: 최소 0.704  최대 0.788  표준편차 0.020


## 정리

- **위험은 "한 번 봤다"가 아니라 "여러 번 봤다"입니다.**
  후보가 하나면 부풀림이 없고, 100개면 실제 0.750이 0.854로 보고됩니다.
- **시험 데이터가 작을수록 위험합니다.** 100개일 때가 5,000개일 때보다
  일곱 배 심합니다.
- 그래서 규칙은 —
  1. **모든 판단은 검증 데이터로.** 층·학습률·조기종료·문턱값·증강 전부.
  2. **시험 데이터는 마지막에 한 번만.** 그 숫자가 보고할 숫자입니다.
  3. 그 숫자를 보고 고쳤다면 **이미 오염된 것**입니다.
  4. **시험 데이터를 넉넉히 두십시오.**

### 연습

1. 후보 수를 5 → 100으로 늘려 가며 부풀림이 어떻게 커지는지 그래프로 그리십시오.
2. `p_true` 를 0.95로 올리면 편향이 커집니까 작아집니까. 왜 그렇습니까.
3. 여러분이 5장 노트북을 돌리며 시험 데이터를 몇 번 봤는지 세어 보고,
   위 표에 대입해 부풀림을 추정하십시오.